# LeetCode 2769 — Find the Maximum Achievable Number bằng Python

Link đề: https://leetcode.com/problems/find-the-maximum-achievable-number/ · Độ khó: **Easy** · Chủ đề: Math, Simulation

Skeleton LeetCode:

```python
class Solution:
    def theMaximumAchievableX(self, num: int, t: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một số `x` gọi là **achievable (đạt được)** nếu sau **tối đa `t` thao tác** (mỗi thao tác: **đồng thời** tăng/giảm `x` đúng 1 **và** tăng/giảm `num` đúng 1) thì `x` bằng `num`; tìm `x` **lớn nhất**.
2. Học cách **mô hình hoá bằng một đại lượng duy nhất**: khoảng cách `gap = x − num`. Mỗi thao tác đổi `gap` một lượng `−2`, `0` hoặc `+2`.
3. Đi từ **thử từng `x` + mô phỏng** → **vét cạn bằng BFS trên mọi trạng thái** (oracle) → **mô phỏng ngược t lần** → **công thức `num + 2t`** → **liệt kê tập mọi `x` đạt được**.
4. Hiểu **bất biến chẵn lẻ** của `gap` (luôn cùng tính chẵn lẻ) và chứng minh `x_max = num + 2t`.
5. Tránh các lỗi: nghĩ chỉ một số di chuyển (`num + t`), lẫn lộn max / min, và quên rằng `x` chỉ đạt được khi `gap` chẵn.

## 2. Đề bài & ví dụ

Cho hai số nguyên `num` và `t`.

Một số nguyên `x` được gọi là **achievable** nếu nó có thể trở nên **bằng `num`** sau khi áp dụng thao tác sau **tối đa `t` lần**:

- **Tăng hoặc giảm `x` đi 1, đồng thời tăng hoặc giảm `num` đi 1.**

Trả về **số achievable lớn nhất**. Có thể chứng minh luôn tồn tại ít nhất một số achievable.

| Ví dụ | num, t | Output | Giải thích |
|---|---|---|---|
| 1 | `4`, `1` | `6` | Với `x = 6`: **giảm `x` đi 1, tăng `num` lên 1** ⇒ `x = 5`, `num = 5` bằng nhau. Không có số nào lớn hơn 6 đạt được. |
| 2 | `3`, `2` | `7` | Với `x = 7`: thao tác 1: giảm `x`, tăng `num` ⇒ `x = 6, num = 4`; thao tác 2: lại như vậy ⇒ `x = 5, num = 5` bằng nhau. |

Constraints:

- `1 <= num, t <= 50`

In [1]:
# Chạy thử bằng tay ví dụ 2: num = 3, t = 2, x = 7
x, num, t = 7, 3, 2
for step in range(1, t + 1):
    x -= 1          # giảm x
    num += 1        # tăng num (cùng lúc)
    print(f"thao tác {step}: x = {x}, num = {num}, gap = {x - num}")
print("bằng nhau?", x == num)

thao tác 1: x = 6, num = 4, gap = 2
thao tác 2: x = 5, num = 5, gap = 0
bằng nhau? True


## 3. Kiến thức nền

### 3.1. Khoảng cách `gap = x − num`

Thay vì theo dõi hai số, ta theo dõi **hiệu** `gap = x − num`. Mục tiêu của đề là đưa `gap` về `0` (khi `x == num`).

- Giảm `x` đi 1: `gap` giảm 1.
- Tăng `num` lên 1: `gap` giảm thêm 1.
- Vậy "giảm `x`, tăng `num`" làm `gap` giảm **2**.

In [2]:
x, num = 7, 3
gap = x - num
print("gap ban đầu:", gap)
x, num = x - 1, num + 1
print("sau 1 thao tác (x-1, num+1):", x - num)     # 2 nhỏ hơn

gap ban đầu: 4
sau 1 thao tác (x-1, num+1): 2


### 3.2. Bốn cách phối hợp một thao tác và tác động lên `gap`

Mỗi thao tác chọn `dx ∈ {−1, +1}` cho `x` và `dn ∈ {−1, +1}` cho `num` **cùng lúc** (không được bỏ qua một số nào). Khi đó `gap` đổi `dx − dn`:

| `dx` | `dn` | `gap` đổi |
|---|---|---|
| −1 | +1 | **−2** |
| −1 | −1 | 0 |
| +1 | +1 | 0 |
| +1 | −1 | **+2** |

In [3]:
for dx in (-1, 1):
    for dn in (-1, 1):
        print(f"dx={dx:+d}, dn={dn:+d} -> gap đổi {dx - dn:+d}")

dx=-1, dn=-1 -> gap đổi +0
dx=-1, dn=+1 -> gap đổi -2
dx=+1, dn=-1 -> gap đổi +2
dx=+1, dn=+1 -> gap đổi +0


### 3.3. Tính chẵn lẻ (parity) bất biến

Vì `gap` chỉ đổi `−2, 0, +2` (đều chẵn), **tính chẵn lẻ của `gap` không bao giờ thay đổi**. Muốn `x == num` (`gap = 0`, chẵn) thì `gap` ban đầu **phải chẵn**. Đó là lý do một số như `num + 1` (gap lẻ) **không** đạt được, còn `num + 2`, `num + 4`, ... thì có thể.

In [4]:
for gap in range(-4, 5):
    print(gap, "chẵn" if gap % 2 == 0 else "lẻ", "-> có thể về 0" if gap % 2 == 0 else "-> không bao giờ về 0")

-4 chẵn -> có thể về 0
-3 lẻ -> không bao giờ về 0
-2 chẵn -> có thể về 0
-1 lẻ -> không bao giờ về 0
0 chẵn -> có thể về 0
1 lẻ -> không bao giờ về 0
2 chẵn -> có thể về 0
3 lẻ -> không bao giờ về 0
4 chẵn -> có thể về 0


### 3.4. `range(start, stop, step)` và tập hợp bằng `set` / biểu thức sinh

`range(a, b, 2)` sinh `a, a + 2, ...` (dừng trước `b`). Ta dùng nó để liệt kê các `x` có cùng tính chẵn lẻ với `num`. `max(...)` lấy phần tử lớn nhất.

In [5]:
num, t = 3, 2
xs = list(range(num - 2 * t, num + 2 * t + 1, 2))
print(xs)                 # [-1, 1, 3, 5, 7]
print(max(xs))            # 7
print(len(xs), 2 * t + 1) # 5 5

[-1, 1, 3, 5, 7]
7
5 5


### 3.5. Mô phỏng bằng vòng `for` và dừng sớm

Trong vòng lặp ta kiểm tra điều kiện dừng: `if x == num: break` kết thúc sớm khi hai số đã bằng nhau (không cần dùng hết `t` thao tác, vì đề cho "tối đa `t`").

In [6]:
x, num, t = 5, 3, 5
steps = 0
for _ in range(t):
    if x == num:
        break
    x -= 1
    num += 1
    steps += 1
print(x, num, "số thao tác dùng:", steps)     # 4 4 1

4 4 số thao tác dùng: 1


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Tính tay ví dụ 2: `num = 3`, `t = 2`.

Ta muốn `x` **càng lớn càng tốt**, nên `gap = x − 3` lớn. Mỗi thao tác tốt nhất làm `gap` **giảm 2**. Sau `t = 2` thao tác tối đa giảm được `4`. Muốn về đúng `0` thì `gap` ban đầu tối đa là `4`, tức `x = 3 + 4 = 7`.

| `x` | `gap` ban đầu | có thể về 0 trong ≤ 2 thao tác? | lý do |
|---|---|---|---|
| 9 | 6 | không | cần giảm 6, tối đa giảm 4 |
| 8 | 5 | không | gap lẻ (và cũng quá lớn) |
| **7** | **4** | **có** | 2 thao tác × (−2) = −4 |
| 6 | 3 | không | gap lẻ |
| 5 | 2 | có | 1 thao tác |
| 3 | 0 | có | 0 thao tác |

**Nhận xét then chốt:** số lớn nhất đạt được là `num + 2t`. Đây là giá trị lớn nhất mà `gap` có thể "tiêu thụ" hết bằng `t` bước, mỗi bước giảm tối đa `2`.

In [7]:
num, t = 3, 2
for x in range(9, 2, -1):
    gap = x - num
    ok = gap % 2 == 0 and gap <= 2 * t
    print(f"x={x}: gap={gap}, đạt được? {ok}")

x=9: gap=6, đạt được? False
x=8: gap=5, đạt được? False
x=7: gap=4, đạt được? True
x=6: gap=3, đạt được? False
x=5: gap=2, đạt được? True
x=4: gap=1, đạt được? False
x=3: gap=0, đạt được? True


## 5. Cách 1 — Brute Force / baseline: thử từng `x`, mô phỏng tham lam

**Ý tưởng:** viết hàm `achievable(x, num, t)`: mô phỏng tối đa `t` thao tác, mỗi thao tác chọn **tốt nhất cho việc thu hẹp khoảng cách** là `x − 1`, `num + 1` (khi `x > num`); kiểm tra cuối cùng `x == num`. Rồi thử `x` từ một giá trị **lớn thừa** giảm dần; số đầu tiên đạt được là đáp án.

Ta cố ý thử `x` bắt đầu từ `num + 3t` (lớn hơn xa đáp án thực) để cách này **không phụ thuộc vào kết luận** của ta.

In [8]:
from typing import List

class SolutionTry:
    def theMaximumAchievableX(self, num: int, t: int) -> int:
        def achievable(x: int, n: int) -> bool:
            for _ in range(t):
                if x == n:
                    break
                if x > n:
                    x -= 1
                    n += 1
                else:
                    x += 1
                    n -= 1
            return x == n

        for x in range(num + 3 * t, num - 1, -1):
            if achievable(x, num):
                return x
        return num                     # x = num luôn đạt được với 0 thao tác

In [9]:
sol = SolutionTry()
print(sol.theMaximumAchievableX(4, 1))   # 6
print(sol.theMaximumAchievableX(3, 2))   # 7

6
7


**Độ phức tạp:** ta thử các ứng viên từ `num + 3t` xuống `num + 2t`, tức `t + 1` ứng viên, mỗi ứng viên mô phỏng `O(t)` ⇒ Time `O(t²)`, Space `O(1)`.

**Nút thắt:** ta dò từng `x`, trong khi các `x` đạt được có quy luật rõ ràng (`gap` chẵn, `|gap| <= 2t`). Ta có thể bỏ hẳn việc dò.

## 6. Ý tưởng thuật toán / bước đột phá

**Đột phá 1 — chỉ cần quan tâm `gap`.** Mỗi thao tác đổi `gap` một lượng thuộc `{−2, 0, +2}`. Sau tối đa `t` thao tác, `gap` có thể thay đổi tổng cộng một lượng chẵn có độ lớn tối đa `2t`.

**Đột phá 2 — điều kiện cần và đủ.** `x` đạt được **khi và chỉ khi** `gap = x − num` chẵn và `|gap| <= 2t`:

- (Cần) `gap` về `0` đòi hỏi tổng thay đổi bằng `−gap`: tính chẵn lẻ bất biến ⇒ `gap` chẵn; mỗi bước đổi tối đa 2 ⇒ `|gap| <= 2t`.
- (Đủ) Nếu `gap = 2m` với `0 <= m <= t` thì dùng `m` thao tác `x − 1, num + 1` (giảm `2` mỗi lần) để `gap` về `0`; còn `gap = −2m` thì dùng `x + 1, num − 1`.

**Đột phá 3 — số lớn nhất.** Số `x` lớn nhất thoả điều kiện là `num + 2t` (với `gap = 2t`).

**Đột phá 4 — "gặp nhau ở giữa".** Cùng kết quả nhìn theo cách khác: sau `t` thao tác hai số gặp nhau tại `v = num + t` (`num` tăng `t`, `x` giảm `t`), nên `x = v + t = num + 2t`.

In [10]:
num, t = 3, 2
meet = num + t
x_max = meet + t
print("điểm gặp nhau:", meet, " x lớn nhất:", x_max, " công thức:", num + 2 * t)

điểm gặp nhau: 5  x lớn nhất: 7  công thức: 7


## 7. Các cách tiếp theo

### Cách 2 — Vét cạn đúng định nghĩa bằng BFS trên mọi trạng thái `(x, num)` (oracle)

**Ý tưởng:** không dùng bất kỳ nhận xét nào ở trên. Với mỗi ứng viên `x`, thử **mọi** thao tác (4 cách phối hợp) trong tối đa `t` bước và xem có đạt `x == num` không. Cài đặt bằng tập các trạng thái đạt tới sau mỗi bước (tránh lặp lại).

Vì độ phức tạp lớn, ta chỉ dùng cách này làm **oracle** trên `num`, `t` nhỏ để chứng minh các cách khác đúng.

In [11]:
class SolutionBFS:
    def theMaximumAchievableX(self, num: int, t: int) -> int:
        def achievable(x0: int) -> bool:
            frontier = {(x0, num)}
            for _ in range(t + 1):                      # kiểm tra trước khi thao tác, tối đa t thao tác
                if any(a == b for a, b in frontier):
                    return True
                frontier = {(a + dx, b + dn)
                            for a, b in frontier
                            for dx in (-1, 1)
                            for dn in (-1, 1)}
            return False

        best = None
        for x in range(num - 4 * t, num + 4 * t + 1):
            if achievable(x):
                best = x if best is None else max(best, x)
        return best

In [12]:
sol = SolutionBFS()
print(sol.theMaximumAchievableX(4, 1))   # 6
print(sol.theMaximumAchievableX(3, 2))   # 7

6
7


**Độ phức tạp:** số trạng thái mỗi bước `O(t²)`, `t` bước, `8t + 1` ứng viên ⇒ khoảng `O(t⁴)`. Chỉ dùng cho `t` nhỏ.

### Cách 3 — Mô phỏng ngược: chạy `t` thao tác cộng dồn

**Từng đoạn:** xuất phát từ trạng thái "đã bằng nhau" rồi đi ngược: mỗi thao tác ngược làm `x` tăng thêm `2` **so với `num`** (vì `x` tăng 1, `num` giảm 1). Sau `t` thao tác, `x` hơn `num` đúng `2t`.

Ta mô phỏng bằng biến `gap`: bắt đầu `gap = 0`, `t` lần `gap += 2`, đáp án `num + gap`.

In [13]:
class SolutionLoop:
    def theMaximumAchievableX(self, num: int, t: int) -> int:
        gap = 0
        for _ in range(t):
            gap += 2          # mỗi thao tác "mở" khoảng cách thêm tối đa 2
        return num + gap

In [14]:
sol = SolutionLoop()
print(sol.theMaximumAchievableX(4, 1))   # 6
print(sol.theMaximumAchievableX(3, 2))   # 7

6
7


**Độ phức tạp:** Time `O(t)`, Space `O(1)`.

### Cách 4 — Công thức `num + 2 * t` (code nộp)

Gộp vòng lặp của Cách 3: `t` lần cộng `2` bằng `2 * t`.

In [15]:
class SolutionFormula:
    def theMaximumAchievableX(self, num: int, t: int) -> int:
        return num + 2 * t

In [16]:
sol = SolutionFormula()
print(sol.theMaximumAchievableX(4, 1))   # 6
print(sol.theMaximumAchievableX(3, 2))   # 7

6
7


**Độ phức tạp:** Time `O(1)`, Space `O(1)`.

### Cách 5 — Liệt kê **tập** mọi `x` đạt được rồi lấy `max`

**Từng đoạn:** dùng điều kiện cần và đủ: `x = num + gap` với `gap` chẵn, `|gap| <= 2t`. Dùng `range(−2t, 2t + 1, 2)` sinh các `gap`, rồi `max`. Cách này cho thấy **toàn bộ** tập đáp án (không chỉ số lớn nhất), có ích cho các biến thể (số nhỏ nhất, số lượng).

In [17]:
class SolutionSet:
    def theMaximumAchievableX(self, num: int, t: int) -> int:
        achievable = {num + gap for gap in range(-2 * t, 2 * t + 1, 2)}
        return max(achievable)

In [18]:
sol = SolutionSet()
print(sol.theMaximumAchievableX(4, 1))   # 6
print(sol.theMaximumAchievableX(3, 2))   # 7
print(sorted({3 + gap for gap in range(-4, 5, 2)}))   # tập achievable với num=3, t=2: [-1, 1, 3, 5, 7]

6
7
[-1, 1, 3, 5, 7]


**Độ phức tạp:** Time `O(t)`, Space `O(t)`.

## 8. TRACE từng bước

Trace Cách 1 (thử từng `x`) cho `num = 4`, `t = 1`, thử `x = 9, 8, 7, 6`:

| ứng viên `x` | `gap` | thao tác tham lam (tối đa 1) | kết quả | đạt được? |
|---|---|---|---|---|
| 7 | 3 | `x = 6, num = 5` | còn lệch 1 | không (hết thao tác) |
| 6 | 2 | `x = 5, num = 5` | bằng nhau | **có** ⇒ trả 6 |

Trace công thức: `num + 2t = 4 + 2 = 6`.

In [19]:
def trace_try(num, t):
    print(f"num={num}, t={t}")
    for x0 in range(num + 3 * t, num - 1, -1):
        x, n = x0, num
        for _ in range(t):
            if x == n:
                break
            if x > n:
                x, n = x - 1, n + 1
            else:
                x, n = x + 1, n - 1
        ok = x == n
        print(f"  thử x={x0}: gap={x0 - num}, sau tối đa {t} thao tác: x={x}, num={n}, đạt được? {ok}")
        if ok:
            print("  => đáp án", x0)
            return x0

trace_try(4, 1)
trace_try(3, 2)

num=4, t=1
  thử x=7: gap=3, sau tối đa 1 thao tác: x=6, num=5, đạt được? False
  thử x=6: gap=2, sau tối đa 1 thao tác: x=5, num=5, đạt được? True
  => đáp án 6
num=3, t=2
  thử x=9: gap=6, sau tối đa 2 thao tác: x=7, num=5, đạt được? False
  thử x=8: gap=5, sau tối đa 2 thao tác: x=6, num=5, đạt được? False
  thử x=7: gap=4, sau tối đa 2 thao tác: x=5, num=5, đạt được? True
  => đáp án 7


7

In [20]:
def trace_gap(num, t):
    gap = 0
    print(f"bắt đầu gap = {gap}")
    for step in range(1, t + 1):
        gap += 2
        print(f"thao tác ngược {step}: gap = {gap}  (x = num + gap = {num + gap})")
    return num + gap

trace_gap(3, 2)

bắt đầu gap = 0
thao tác ngược 1: gap = 2  (x = num + gap = 5)
thao tác ngược 2: gap = 4  (x = num + gap = 7)


7

## 9. Code nộp LeetCode

Dùng Cách 4 (công thức `O(1)`).

In [21]:
from typing import List

class Solution:
    def theMaximumAchievableX(self, num: int, t: int) -> int:
        # mỗi thao tác thu hẹp tối đa 2 đơn vị khoảng cách giữa x và num
        return num + 2 * t

In [22]:
print(Solution().theMaximumAchievableX(4, 1))   # 6
print(Solution().theMaximumAchievableX(3, 2))   # 7

6
7


## 10. Vì sao thuật toán đúng

Gọi `gap = x − num`. Mỗi thao tác đổi `x` bởi `dx ∈ {±1}` và `num` bởi `dn ∈ {±1}`, nên `gap` đổi `dx − dn ∈ {−2, 0, +2}`.

**Cận trên.** Sau `k <= t` thao tác `gap` đã đổi tổng cộng không quá `2k` về giá trị tuyệt đối, tức `|gap_cuối − gap_đầu| <= 2k <= 2t`. Để `x` bằng `num` ở cuối (`gap_cuối = 0`) ta cần `gap_đầu = x − num <= 2t`, hay `x <= num + 2t`. Vậy không có số nào lớn hơn `num + 2t` đạt được.

**Đạt được cận trên.** Lấy `x = num + 2t`. Thực hiện `t` lần thao tác "giảm `x` đi 1, tăng `num` lên 1": sau mỗi lần `gap` giảm đúng `2`, sau `t` lần `gap = 2t − 2t = 0` nên `x = num` ✓. Vậy `num + 2t` đạt được.

Hai điều trên cho thấy `num + 2t` là số lớn nhất.

**Bổ sung — tập đầy đủ.** Số `x` đạt được ⇔ `gap` chẵn và `|gap| <= 2t`: chiều thuận chứng minh ở trên; chiều ngược dùng `|gap|/2 <= t` thao tác cùng hướng. Điều này khớp với Cách 5, và Cách 2 (BFS không dùng nhận xét nào) là bằng chứng thực nghiệm độc lập, ta kiểm ở mục 12.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Thử từng `x` + mô phỏng | `O(t²)` | `O(1)` | `~3t` ứng viên × `O(t)` |
| 2. BFS mọi trạng thái | `O(t⁴)` | `O(t²)` | chỉ để làm oracle |
| 3. Vòng lặp `gap += 2` | `O(t)` | `O(1)` | mô phỏng ngược |
| 4. Công thức | `O(1)` | `O(1)` | **code nộp** |
| 5. Liệt kê tập achievable | `O(t)` | `O(t)` | có cả tập đáp án |

Với `t <= 50` mọi cách đều rất nhanh. Bài này là ví dụ điển hình: **tìm đại lượng bất biến / đại lượng đơn giản (`gap`) giúp biến mô phỏng thành công thức**.

## 12. Bộ test

### 12.1. Ví dụ của đề và edge case

Các ca quan trọng: `t = 1`; `num = 1`; `t = 50`; `num = 50`; cả hai lớn nhất.

In [23]:
sols = [SolutionTry(), SolutionBFS(), SolutionLoop(), SolutionFormula(), SolutionSet(), Solution()]
fast = [s for s in sols if not isinstance(s, SolutionBFS)]    # BFS quá chậm với t lớn
cases = [
    ("ví dụ 1", 4, 1, 6),
    ("ví dụ 2", 3, 2, 7),
    ("nhỏ nhất", 1, 1, 3),
    ("num lớn, t nhỏ", 50, 1, 52),
    ("num nhỏ, t lớn", 1, 50, 101),
    ("lớn nhất", 50, 50, 150),
    ("bằng nhau", 7, 7, 21),
]
for name, num, t, exp in cases:
    for s in fast:
        got = s.theMaximumAchievableX(num, t)
        assert got == exp, (name, type(s).__name__, got, exp)
    print("PASS", name, "->", exp)

PASS ví dụ 1 -> 6
PASS ví dụ 2 -> 7
PASS nhỏ nhất -> 3
PASS num lớn, t nhỏ -> 52
PASS num nhỏ, t lớn -> 101
PASS lớn nhất -> 150
PASS bằng nhau -> 21


### 12.2. Oracle BFS trên miền nhỏ (không dựa vào nhận xét nào)

Với `1 <= num <= 8`, `1 <= t <= 4`, BFS thử **mọi** thao tác. Mọi cách còn lại phải cho đúng cùng kết quả.

In [24]:
bfs = SolutionBFS()
for num in range(1, 9):
    for t in range(1, 5):
        exp = bfs.theMaximumAchievableX(num, t)
        for s in fast:
            assert s.theMaximumAchievableX(num, t) == exp, (num, t, type(s).__name__)
print("32 cặp (num, t) nhỏ: mọi cách khớp BFS vét cạn")

32 cặp (num, t) nhỏ: mọi cách khớp BFS vét cạn


### 12.3. Toàn miền của đề

Chỉ có `50 × 50 = 2500` cặp; ta kiểm **tất cả** các cách nhanh đối chiếu với Cách 1 (mô phỏng tham lam từng ứng viên) và kiểm tra các tính chất.

In [25]:
for num in range(1, 51):
    for t in range(1, 51):
        exp = SolutionTry().theMaximumAchievableX(num, t)
        assert exp == num + 2 * t
        for s in fast:
            assert s.theMaximumAchievableX(num, t) == exp, (num, t, type(s).__name__)
        # tính chất: cùng tính chẵn lẻ với num và hơn num đúng 2t
        assert (exp - num) % 2 == 0 and exp - num == 2 * t
print("2500 cặp (num, t): mọi cách khớp, đáp án = num + 2t")

2500 cặp (num, t): mọi cách khớp, đáp án = num + 2t


### 12.4. Kiểm cả tập các `x` đạt được bằng BFS

Với miền nhỏ, tập `x` mà BFS xác nhận phải đúng bằng `{num + gap : gap chẵn, |gap| <= 2t}`.

In [26]:
def achievable_bfs(x0, num, t):
    frontier = {(x0, num)}
    for _ in range(t + 1):
        if any(a == b for a, b in frontier):
            return True
        frontier = {(a + dx, b + dn) for a, b in frontier for dx in (-1, 1) for dn in (-1, 1)}
    return False

for num in range(1, 7):
    for t in range(1, 5):
        bfs_set = {x for x in range(num - 4 * t, num + 4 * t + 1) if achievable_bfs(x, num, t)}
        rule_set = {num + gap for gap in range(-2 * t, 2 * t + 1, 2)}
        assert bfs_set == rule_set, (num, t, sorted(bfs_set), sorted(rule_set))
print("tập achievable từ BFS = tập theo quy tắc (gap chẵn, |gap| <= 2t)")

tập achievable từ BFS = tập theo quy tắc (gap chẵn, |gap| <= 2t)


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ có một số di chuyển (`num + t`)

In [27]:
def sai_mot_so_di_chuyen(num: int, t: int) -> int:
    return num + t            # sai: coi như chỉ num hoặc chỉ x thay đổi

print("sai :", sai_mot_so_di_chuyen(4, 1))                    # 5
print("đúng:", Solution().theMaximumAchievableX(4, 1))        # 6

sai : 5
đúng: 6


**Vì sao sai:** mỗi thao tác thay đổi **cả hai số cùng lúc**, `x` giảm 1 **và** `num` tăng 1, nên khoảng cách thu hẹp **2** mỗi lần chứ không phải 1. Ví dụ 1 cho `6`, không phải `5`.

### Lỗi 2: Lẫn lộn lớn nhất với nhỏ nhất

In [28]:
def sai_nho_nhat(num: int, t: int) -> int:
    return num - 2 * t        # đây là số achievable NHỎ nhất

print("sai :", sai_nho_nhat(3, 2))                            # -1
print("đúng:", Solution().theMaximumAchievableX(3, 2))        # 7

sai : -1
đúng: 7


**Vì sao sai:** `num − 2t` cũng đạt được (đối xứng) nhưng là số **nhỏ nhất**. Đề hỏi **lớn nhất**: `num + 2t`. Chú ý rằng `x` có thể nhỏ hơn `num`; đối xứng giữa hai hướng nhắc ta đọc kỹ đề.

### Lỗi 3: Cho rằng mọi `x` trong khoảng đều đạt được

In [29]:
def sai_moi_x(num: int, x: int, t: int) -> bool:
    return abs(x - num) <= 2 * t          # sai: thiếu điều kiện gap chẵn

print("sai   (x = 4, num = 3, t = 2):", sai_moi_x(3, 4, 2))                      # True
print("đúng  :", (4 - 3) % 2 == 0 and abs(4 - 3) <= 2 * 2)                       # False

sai   (x = 4, num = 3, t = 2): True
đúng  : False


**Vì sao sai:** `gap = 1` lẻ; mỗi thao tác đổi `gap` một số **chẵn**, nên `gap` không thể về `0`. Chỉ các `x` có `gap` chẵn mới đạt được. (Với câu hỏi "lớn nhất", điều này không làm đổi đáp án vì `num + 2t` có `gap = 2t` chẵn, nhưng quan trọng khi đếm hoặc kiểm tra một `x` cụ thể.)

### Lỗi 4: Cộng `2` thay vì `2 * t`

In [30]:
def sai_quen_nhan_t(num: int, t: int) -> int:
    return num + 2           # sai: bỏ quên t

print("sai :", sai_quen_nhan_t(3, 2))                        # 5
print("đúng:", Solution().theMaximumAchievableX(3, 2))      # 7

sai : 5
đúng: 7


**Vì sao sai:** mỗi thao tác thu hẹp tối đa 2; có `t` thao tác nên thu hẹp tối đa `2t`.

### Lỗi 5: Tưởng phải dùng **đúng** `t` thao tác

Đề cho "**tối đa** `t` lần". Với `x = num` ta cần `0` thao tác. Với công thức `num + 2t`, ta dùng đúng `t` thao tác. Không có yêu cầu dùng đủ `t` lần với mọi `x`.

In [31]:
print(Solution().theMaximumAchievableX(5, 3))      # 11: dùng đúng 3 thao tác

11


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Khi mỗi thao tác tác động lên **hai đại lượng**, thử gộp chúng thành **một đại lượng** (hiệu hoặc tổng): ở đây `gap = x − num`.
- Hãy tìm **bất biến** (ở đây: tính chẵn lẻ của `gap`) và **tốc độ biến thiên tối đa** (mỗi bước ±2) ⇒ cận trên `2t`.
- Đề kiểu "tối đa `k` lần thao tác, tìm giá trị lớn nhất / nhỏ nhất" thường có đáp án dạng `giá_trị_đầu ± tốc_độ × k`.
- Khi chưa chắc, viết mô phỏng / BFS cho `t` nhỏ (làm oracle), in vài giá trị, rồi đoán công thức.
- Kiểm tra bằng ví dụ nhỏ: `t = 1` cho `num + 2`; `t = 2` cho `num + 4`.

## 15. Biến thể của bài

1. **Số achievable nhỏ nhất:** `num − 2t`.
2. **Số lượng số achievable:** `2t + 1` (các `gap` chẵn từ `−2t` tới `2t`).
3. **Mỗi thao tác chỉ đổi `x` đi 1 (`num` đứng yên):** đáp án `num + t`.
4. **Mỗi thao tác đổi `x` đi `a` và `num` đi `b`** (`a`, `b` cố định): `gap` đổi tối đa `a + b` ⇒ đáp án `num + t(a + b)`.
5. **Phải dùng đúng `t` thao tác** (không phải "tối đa"): vì có thao tác `(+1, +1)` hoặc `(−1, −1)` giữ nguyên `gap`, ta có thể "đứng yên" `gap` ở các bước thừa, nên tập `x` đạt được và đáp án `num + 2t` không đổi.
6. **Đưa hai số `x`, `y` về bằng nhau** với các phép tăng/giảm khác nhau: dạng bài "điểm gặp nhau".

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử từng `x` | mô phỏng tham lam từng ứng viên | `O(t²)` | `O(1)` | khi chưa thấy quy luật |
| 2. BFS mọi trạng thái | thử mọi thao tác | `O(t⁴)` | `O(t²)` | **oracle**, `t` rất nhỏ |
| 3. Vòng `gap += 2` | mô phỏng ngược | `O(t)` | `O(1)` | bước trung gian trước công thức |
| 4. Công thức | `num + 2t` | `O(1)` | `O(1)` | **code nộp** |
| 5. Liệt kê tập | `gap` chẵn, `|gap| <= 2t` | `O(t)` | `O(t)` | cần cả tập đáp án |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `5 + 2 * 3`, `[x for x in range(-2, 3, 2)]`, `(7 - 3) % 2`, `min(range(1, 10, 3))`.

**Bài 2 (tính tay).** Với `num = 6`, `t = 3`: liệt kê mọi `x` đạt được và chỉ ra số nhỏ nhất, lớn nhất.

**Bài 3 (điền TODO).** Hoàn thành `max_achievable_todo(num, t)` bằng vòng lặp (Cách 3).

In [32]:
def max_achievable_todo(num, t):
    gap = 0
    # TODO 1: lặp t lần, mỗi lần gap += 2
    # TODO 2: return num + gap
    return None

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi.

**Bài 4 (tự viết lại).** Viết `is_achievable(x, num, t)` trả `True/False` với điều kiện (gap chẵn, `|gap| <= 2t`) mà không nhìn code mẫu. Kiểm tra bằng BFS (`achievable_bfs`) trên miền nhỏ.

**Bài 5 (biến thể / mini challenge).** (a) Viết `min_achievable(num, t)` và `count_achievable(num, t)`. (b) Mỗi thao tác đổi `x` đi 1 và `num` đi 2 (cả hai **bắt buộc** đổi, hướng tuỳ chọn): viết `max_achievable_12(num, t)`; kiểm tra bằng BFS.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `5 + 6 = 11`; `range(-2, 3, 2) = -2, 0, 2`; `4 % 2 = 0`; `range(1, 10, 3) = 1, 4, 7` nên min `1`. Bài 2: `gap` chẵn từ `−6` tới `6`.
- **Mức 2 (vừa):** Bài 3: `for _ in range(t): gap += 2`. Bài 4: `gap = x - num; return gap % 2 == 0 and abs(gap) <= 2 * t`.
- **Mức 3 (sâu):** Bài 5(a): `min = num − 2t`, số lượng `2t + 1`. Bài 5(b): mỗi thao tác đổi `gap` một lượng `dx − dn` với `dx = ±1`, `dn = ±2`, tức thuộc `{−3, −1, 1, 3}`; mức giảm tối đa mỗi bước là `3` nên đáp án `num + 3t`. Hãy kiểm bằng BFS (đổi `(-1, 1)` thành `(-2, 2)` cho `dn`).

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [33]:
# Bài 1
assert 5 + 2 * 3 == 11
assert [x for x in range(-2, 3, 2)] == [-2, 0, 2]
assert (7 - 3) % 2 == 0
assert min(range(1, 10, 3)) == 1

# Bài 2: num = 6, t = 3 -> gap trong {-6, -4, -2, 0, 2, 4, 6}
xs = sorted(6 + gap for gap in range(-6, 7, 2))
assert xs == [0, 2, 4, 6, 8, 10, 12]
assert min(xs) == 0 and max(xs) == 12 == Solution().theMaximumAchievableX(6, 3)

# Bài 3
def max_achievable_todo_ans(num, t):
    gap = 0
    for _ in range(t):
        gap += 2
    return num + gap

assert max_achievable_todo_ans(4, 1) == 6
assert max_achievable_todo_ans(3, 2) == 7

# Bài 4
def is_achievable(x, num, t):
    gap = x - num
    return gap % 2 == 0 and abs(gap) <= 2 * t

for num in range(1, 6):
    for t in range(1, 4):
        for x in range(num - 4 * t, num + 4 * t + 1):
            assert is_achievable(x, num, t) == achievable_bfs(x, num, t), (x, num, t)

# Bài 5(a)
def min_achievable(num, t):
    return num - 2 * t

def count_achievable(num, t):
    return 2 * t + 1

for num in range(1, 6):
    for t in range(1, 4):
        s = {x for x in range(num - 4 * t, num + 4 * t + 1) if achievable_bfs(x, num, t)}
        assert min(s) == min_achievable(num, t)
        assert len(s) == count_achievable(num, t)

# Bài 5(b): dx = ±1, dn = ±2 (bắt buộc đổi cả hai); gap đổi dx - dn thuộc {-3, -1, 1, 3}
def achievable_bfs_12(x0, num, t):
    frontier = {(x0, num)}
    for _ in range(t + 1):
        if any(a == b for a, b in frontier):
            return True
        frontier = {(a + dx, b + dn) for a, b in frontier for dx in (-1, 1) for dn in (-2, 2)}
    return False

def max_achievable_12(num, t):
    return num + 3 * t

for num in range(1, 6):
    for t in range(1, 4):
        best = max(x for x in range(num - 4 * t, num + 5 * t + 1) if achievable_bfs_12(x, num, t))
        assert best == max_achievable_12(num, t), (num, t, best)
print("Tất cả đáp án đúng")

Tất cả đáp án đúng


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (gộp hai đại lượng thành khoảng cách, tìm tốc độ thu hẹp tối đa)

```python
from typing import List

class Solution:
    def theMaximumAchievableX(self, num: int, t: int) -> int:
        # gap = x - num; mỗi thao tác thu hẹp gap tối đa 2; t thao tác -> tối đa 2t
        return num + 2 * t
```

**Checklist**

- [ ] Hiểu mỗi thao tác đổi **cả `x` lẫn `num`** (nên gap thay đổi tối đa 2, không phải 1)?
- [ ] Biết `gap` luôn cùng tính chẵn lẻ (bất biến) nên `x` đạt được cần `gap` chẵn?
- [ ] Đáp án **lớn nhất** là `num + 2t` (nhỏ nhất là `num − 2t`)?
- [ ] Hiểu "tối đa `t`" (không bắt buộc dùng đủ)?
- [ ] Đã thử `t = 1`, `num = 1`, `t = 50`?
- [ ] Đã kiểm bằng BFS vét cạn ở miền nhỏ?
- [ ] Signature đúng `theMaximumAchievableX(self, num: int, t: int) -> int`?